<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/standards/standards_based_engineering_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standards-Based Engineering Checks

Use simple public screening checks to flag line velocity, pressure drop, and relief-load issues before a formal standards review.


## Setup

Install imports and define the gas fluid.


In [1]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
from neqsim import jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 26.9 MB/s eta 0:00:00


## Define Design Cases

These inputs represent early design-screening values.


In [2]:
cases = pd.DataFrame({
    "line": ["A", "B", "C"],
    "flow_kg_per_hr": [5000, 12000, 20000],
    "diameter_m": [0.15, 0.20, 0.25],
    "pressure_bara": [60, 80, 100],
    "temperature_C": [20, 25, 30],
})
cases


,line,flow_kg_per_hr,diameter_m,pressure_bara,temperature_C
0,A,5000,0.15,60,20
1,B,12000,0.20,80,25
2,C,20000,0.25,100,30


## Run Public Screening Checks

The limits are educational placeholders and must not replace project-specific standards interpretation.


In [3]:
import math
rows = []
for _, row in cases.iterrows():
    fluid = make_gas(float(row.temperature_C), float(row.pressure_bara))
    density = fluid.getDensity("kg/m3")
    area = math.pi * (row.diameter_m ** 2) / 4.0
    velocity = row.flow_kg_per_hr / 3600.0 / density / area
    pressure_gradient_bar_per_km = 0.02 * velocity ** 2
    rows.append({
        "line": row.line,
        "velocity_m_per_s": velocity,
        "velocity_flag": velocity > 25.0,
        "pressure_gradient_bar_per_km": pressure_gradient_bar_per_km,
        "pressure_drop_flag": pressure_gradient_bar_per_km > 0.5,
    })
checks = pd.DataFrame(rows)
checks


,line,velocity_m_per_s,velocity_flag,pressure_gradient_bar_per_km,pressure_drop_flag
0,A,1.438105,False,0.041363,False
1,B,1.433581,False,0.041103,False
2,C,1.220856,False,0.029810,False


## Summarize Findings

A flag means the case should be reviewed with the governing project standard and a validated calculation method.


In [4]:
checks.assign(any_flag=checks[["velocity_flag", "pressure_drop_flag"]].any(axis=1))


,line,velocity_m_per_s,velocity_flag,pressure_gradient_bar_per_km,pressure_drop_flag,any_flag
0,A,1.438105,False,0.041363,False,False
1,B,1.433581,False,0.041103,False,False
2,C,1.220856,False,0.029810,False,False
